In [13]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


In [14]:
# 1. Project paths

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data" / "raw"
REPORTS_DIR = BASE_DIR / "reports"
FIGURES_DIR = BASE_DIR / "figures"

REPORTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

In [15]:
# 2. Load credit card data

cards = pd.read_csv(DATA_DIR / "credit_cards.csv")

In [16]:
# 3. Define spending scenarios

scenarios = {
    "Average Consumer": {
        "Groceries": 600,
        "Dining": 300,
        "Gas": 180,
        "Travel": 150,
        "Bills": 250,
        "Other": 350
    },

    "Grocery and Dining Consumer": {
        "Groceries": 1000,
        "Dining": 600,
        "Gas": 100,
        "Travel": 100,
        "Bills": 250,
        "Other": 200
    },

    "Frequent Traveler": {
        "Groceries": 400,
        "Dining": 300,
        "Gas": 100,
        "Travel": 1200,
        "Bills": 250,
        "Other": 250
    },

    "Gas and Commuting Consumer": {
        "Groceries": 400,
        "Dining": 200,
        "Gas": 600,
        "Travel": 100,
        "Bills": 250,
        "Other": 250
    }
}


In [17]:
# 4. Define reward rate columns

rate_columns = {
    "Groceries": "groceries_rate",
    "Dining": "dining_rate",
    "Gas": "gas_rate",
    "Travel": "travel_rate",
    "Bills": "bills_rate",
    "Other": "other_rate"
}

In [18]:
# 5. Calculate rewards for each scenario

results = []

for scenario_name, spending in scenarios.items():

    annual_spending = {
        category: monthly_amount * 12
        for category, monthly_amount in spending.items()
    }

    total_annual_spending = sum(annual_spending.values())

    for _, card in cards.iterrows():

        total_rewards = 0

        for category, annual_amount in annual_spending.items():

            rate_column = rate_columns[category]

            reward_rate = card[rate_column]

            category_reward = annual_amount * reward_rate

            total_rewards += category_reward

        annual_fee = card["annual_fee"]

        welcome_bonus = card["welcome_bonus_value"]

        ongoing_net_value = total_rewards - annual_fee

        first_year_net_value = (
            ongoing_net_value + welcome_bonus
        )

        results.append({
            "scenario": scenario_name,
            "card_name": card["card_name"],
            "annual_spending": total_annual_spending,
            "annual_rewards": total_rewards,
            "annual_fee": annual_fee,
            "ongoing_net_value": ongoing_net_value,
            "welcome_bonus_value": welcome_bonus,
            "first_year_net_value": first_year_net_value
        })


In [19]:
# 6. Create results DataFrame

results_df = pd.DataFrame(results)

In [20]:
# 7. Rank cards within each scenario

results_df["scenario_rank"] = (
    results_df.groupby("scenario")["ongoing_net_value"]
    .rank(method="min", ascending=False)
)

results_df = results_df.sort_values(
    ["scenario", "scenario_rank"]
)

In [21]:
# 8. Identify the best card for each scenario

best_cards = (
    results_df
    .sort_values("ongoing_net_value", ascending=False)
    .groupby("scenario")
    .first()
    .reset_index()
)


In [22]:
# 9. Export results

results_df.to_csv(
    REPORTS_DIR / "scenario_analysis_results.csv",
    index=False
)

best_cards.to_csv(
    REPORTS_DIR / "best_card_by_scenario.csv",
    index=False
)

In [23]:
# 10. Print results

print("\n" + "=" * 60)
print("CREDIT CARD SCENARIO ANALYSIS")
print("=" * 60)

for scenario_name in scenarios:

    scenario_results = results_df[
        results_df["scenario"] == scenario_name
    ]

    print(f"\nScenario: {scenario_name}")

    print(
        scenario_results[
            [
                "card_name",
                "annual_rewards",
                "annual_fee",
                "ongoing_net_value",
                "scenario_rank"
            ]
        ].to_string(index=False)
    )



CREDIT CARD SCENARIO ANALYSIS

Scenario: Average Consumer
              card_name  annual_rewards  annual_fee  ongoing_net_value  scenario_rank
Wells Fargo Active Cash           439.2           0              439.2            1.0
      Capital One Savor           435.6           0              435.6            2.0
Chase Freedom Unlimited           383.4           0              383.4            3.0

Scenario: Grocery and Dining Consumer
              card_name  annual_rewards  annual_fee  ongoing_net_value  scenario_rank
      Capital One Savor           654.0           0              654.0            1.0
Wells Fargo Active Cash           540.0           0              540.0            2.0
Chase Freedom Unlimited           513.0           0              513.0            3.0

Scenario: Frequent Traveler
              card_name  annual_rewards  annual_fee  ongoing_net_value  scenario_rank
Wells Fargo Active Cash           600.0           0              600.0            1.0
Chase Freedom

In [24]:
# 11. Create visualization

pivot_results = results_df.pivot(
    index="scenario",
    columns="card_name",
    values="ongoing_net_value"
)

pivot_results.plot(
    kind="bar",
    figsize=(12, 7)
)

plt.title(
    "Credit Card Net Value by Spending Scenario",
    fontsize=16,
    fontweight="bold"
)

plt.xlabel("Consumer Spending Scenario")
plt.ylabel("Annual Net Value ($)")

plt.xticks(rotation=25, ha="right")

plt.legend(
    title="Credit Card",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "scenario_analysis_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()
